# CreditLens — Notebook 04: Statistical Analysis & Concentration
**Objective:** Measure market concentration via Herfindahl-Hirschman Index (HHI), Top-3 exposure, growth correlations, and structural break impact.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df_conc = pd.read_csv('../data/processed/industry_concentration_monthly.csv')
df_conc['date'] = pd.to_datetime(df_conc['reporting_month'] + '-01')
display(df_conc.tail())


### 1. HHI Trajectory Over Time


In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(df_conc['date'], df_conc['hhi_index'], color='#06B6D4', linewidth=2, label='HHI (Points)')
plt.axhline(1500, color='#F59E0B', linestyle='--', label='Moderate Concentration (1,500 pts)')
plt.axhline(1000, color='#10B981', linestyle='--', label='Unconcentrated Threshold (1,000 pts)')
plt.title('Herfindahl-Hirschman Index (HHI) Trajectory (2019 – 2026)')
plt.ylabel('HHI Points')
plt.grid(True, linestyle='--', alpha=0.4)
plt.legend()
plt.show()


### 2. Cross-Sector Growth Correlation Matrix


In [ ]:
df_metrics = pd.read_csv('../data/processed/sector_monthly_metrics.csv')
df_l1 = df_metrics[df_metrics['is_mutually_exclusive'] == True]
pivot_growth = df_l1.pivot(index='reporting_month', columns='sector_name', values='yoy_growth_pct').dropna()

# Select top 7 sectors for clean correlation matrix
top_secs = df_l1.groupby('sector_name')['credit_outstanding_cr'].mean().nlargest(7).index
corr_matrix = pivot_growth[top_secs].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('YoY Growth Rate Correlation Across Leading Sectors')
plt.show()


### 3. Structural Break Analysis (July 2023 Bank-NBFC Merger)


In [ ]:
df_tot = df_metrics[df_metrics['sector_code'] == '2'].sort_values('reporting_date')
pre_merger = df_tot[df_tot['reporting_month'] == '2023-06']['credit_outstanding_cr'].iloc[0]
post_merger = df_tot[df_tot['reporting_month'] == '2023-07']['credit_outstanding_cr'].iloc[0]

jump_cr = post_merger - pre_merger
jump_pct = (jump_cr / pre_merger) * 100

print(f"June 2023 Industry Credit: ₹{pre_merger:,.2f} Cr")
print(f"July 2023 Industry Credit: ₹{post_merger:,.2f} Cr")
print(f"One-Month Level Shift: +₹{jump_cr:,.2f} Cr ({jump_pct:.2f}%)")
